# 制約充足問題（CSP）― お弁当を食べ終わる順序を決定しましょう

## 本章で学ぶこと

本章では、**制約充足問題**（Constraint Satisfaction Problem、略して **CSP**）を扱います。

- 制約充足問題とはどのような問題かを説明します。
- 総当たり（全数探索）では、なぜすぐに扱えなくなるのかを説明します。
- Z3 を用いると、なぜ容易に解けるのかを説明します。

本章で扱うキーワードは「変数」と「制約」の2つだけです。特別な前提知識は不要です。

## 制約充足問題とは

制約充足問題とは、**いくつかの変数に、条件（制約）を満たすように値を決める問題**です。

以下に小さな例を示します。

> 5人の子供（A, B, C, D, E）が、順番にお弁当を食べ終わりました。
> 次の記述から、食べ終わった順序を決定してください。
>
> - A は B より遅く食べ終わりましたが、C よりは早く食べ終わりました。
> - D は E より早く食べ終わりましたが、C よりは遅く食べ終わりました。

この問題では、

- **変数**：5人それぞれの「食べ終わった順位」
- **制約**：「A は B より後」「A は C より前」などの条件

となります。5人の順序を決定し、すべての条件を満たす順序が答えです。

## 条件を整理する

日本語のままでは扱いにくいため、記号で書き直します。

$X \prec Y$ を「X が Y より先に食べ終わった」とします。

1. A は B より遅く、C より早い → $B \prec A \prec C$
2. D は E より早く、C より遅い → $C \prec D \prec E$

2つを結合すると、$B \prec A \prec C \prec D \prec E$ という1本の鎖になります。

つまり、求めるものは「この順序を満たす並び」です。

## まずは総当たりで解きます

最も単純な方法は、**考えられる順序をすべて生成し、条件を満たすものだけを残す**ことです。

5人の並び順は $5! = 120$ 通りしかありません。すべて試しても一瞬で終了します。

In [1]:
from itertools import permutations

kids = ["A", "B", "C", "D", "E"]

# order は「食べ終わった順に並んだ子供の番号」
for order in permutations(range(len(kids))):
    rank = {kids[i]: pos for pos, i in enumerate(order)}   # 各子供の順位
    if rank["B"] < rank["A"] < rank["C"] and rank["C"] < rank["D"] < rank["E"]:
        print(" → ".join(kids[i] for i in order))

B → A → C → D → E


条件を満たす順序は **B → A → C → D → E** の1通りだけでした。

5人であれば一瞬で終了します。では、人数が増えた場合はどうなるでしょうか。並び順の数は $n!$ で増加します。

In [2]:
import math

for n in [5, 8, 10, 12]:
    print(f"{n:2d}人: {math.factorial(n):,}通り")

 5人: 120通り
 8人: 40,320通り
10人: 3,628,800通り
12人: 479,001,600通り


12人では約4億8千万通りです。1秒に100万通りを試せるとしても8分かかります。20人（約2.4×10¹⁸通り）になると、そもそも終了しません。

「すべて試す」という方法は、人数が少し増えるだけで適用できなくなります。本章の後半で、Z3 がこの壁をどのように越えるかを説明します。

### 補足：`itertools.permutations` について

総当たりで用いた `permutations()` は、与えた要素の**すべての並び順**を生成する関数です。

```python
itertools.permutations(iterable, r=None)
```

- `iterable`：並べたい要素
- `r`：何個取り出すか（省略すると全部）

3つの要素なら $3! = 6$ 通りです。

In [3]:
for p in permutations(["A", "B", "C"]):
    print(p)

('A', 'B', 'C')
('A', 'C', 'B')
('B', 'A', 'C')
('B', 'C', 'A')
('C', 'A', 'B')
('C', 'B', 'A')


`permutations()` を用いる際に注意すべき点は3つです。

1. **順序を区別します**：(A, B) と (B, A) は別のものとして扱われます。
2. **重複は考慮しません**：同じ要素が含まれている場合、同じ並びが複数回生成されます。
3. **計算量は $n!$ です**：要素が増えると爆発的に増加します。

小さな問題を目視で確認するには便利ですが、大きな問題には適していません。そこでソルバーを用います。

## Z3 による解法

Z3 は**制約ソルバー**の1つです。条件をそのまま記述すると、条件を満たす値の組み合わせを探索します。

総当たりとの違いは次のとおりです。

| | やること |
|---|---|
| 総当たり | 候補をすべて生成し、1つずつ確認します |
| Z3 | 条件を記述し、探索を Z3 に任せます |

Z3 による解法の手順は常に同じです。

1. 変数を生成します。
2. 制約を追加します。
3. 解を取得します。

In [4]:
from z3 import IntVector, Solver, Distinct, sat

kids = ["A", "B", "C", "D", "E"]

# 各子供の「食べ終わった順位」を表す5つの整数変数（pos[0] が A、pos[1] が B …）
pos = IntVector("pos", 5)
eat_a, eat_b, eat_c, eat_d, eat_e = pos

In [5]:
solver = Solver()

# 1. 順位は 0〜4 のどれか
for p in pos:
    solver.add(p >= 0, p < len(kids))

# 2. 同じ順位の子供はいない
solver.add(Distinct(pos))

# 3. 問題の条件  B ≺ A ≺ C ≺ D ≺ E
solver.add(eat_b < eat_a, eat_a < eat_c)
solver.add(eat_c < eat_d, eat_d < eat_e)

# 4. 解を取り出して、順位の小さい順に並べる
if solver.check() == sat:
    model = solver.model()
    rank = {kid: model[pos[i]].as_long() for i, kid in enumerate(kids)}
    print(" → ".join(sorted(kids, key=lambda kid: rank[kid])))

B → A → C → D → E


総当たりと同じ答えが得られました。ここで、探索範囲が段階的に狭くなる様子に注目してください。

1. `IntVector("pos", 5)` を生成しただけでは、値は任意です → 無限
2. `0 <= p < 5` を追加します → $5^5 = 3125$ 通り
3. `Distinct(pos)` を追加します → $5! = 120$ 通り（総当たりと同じ範囲）
4. 条件 $B \prec A \prec C \prec D \prec E$ を追加します → 1通り

総当たりでは手順3の範囲から開始し、最後まで自らすべて試します。Z3 は条件を先に受け取るため、条件に合わない候補を最初から生成せずに済みます。人数が増えても仕組みは同じです。

## まとめ

| 手法 | 計算量 | 向いている場面 |
|---|---|---|
| `itertools.permutations` | $O(n!)$ | 小さな問題を目視で確認する |
| Z3 | 条件から効率的に探索する | 条件が複雑な問題、人数が多い問題 |

- 制約充足問題は「変数」と「制約」から構成されます。
- 総当たりの計算量は $n!$ であるため、少し大きくなるだけで適用できなくなります。
- Z3 は制約をそのまま記述できるため、問題が変わっても対応しやすいです。